# Feature Extraction with EgoVLP

This notebook:
1. Clones the required repos (EgoVLP, CaptainCook4D feature_extractors, our project).
2. Mounts Google Drive and loads the EgoVLP checkpoint + the resized CaptainCook4D videos.
3. Extracts per-second (1s) EgoVLP video features for every recording and saves them to Drive with the CaptainCook4D `.npz` naming convention.


In [6]:
# Step 1: Clone repositories
!git clone --recursive https://github.com/showlab/EgoVLP /content/egovlp
!git clone --recursive https://github.com/Laio95/aml-2025-mistake-detection.git /content/code

fatal: destination path '/content/egovlp' already exists and is not an empty directory.
Cloning into '/content/code'...
remote: Enumerating objects: 551, done.
remote: Counting objects: 100% (136/136), done.
remote: Compressing objects: 100% (21/21), done.
remote: Total 551 (delta 119), reused 115 (delta 115), pack-reused 415 (from 2)
Receiving objects: 100% (551/551), 2.47 MiB | 2.94 MiB/s, done.
Resolving deltas: 100% (352/352), done.
Submodule 'annotations' (https://github.com/CaptainCook4D/annotations) registered for path 'annotations'
Cloning into '/content/code/annotations'...
remote: Enumerating objects: 152, done.        
remote: Counting objects: 100% (152/152), done.        
remote: Compressing objects: 100% (98/98), done.        
remote: Total 152 (delta 75), reused 108 (delta 46), pack-reused 0 (from 0)        
Receiving objects: 100% (152/152), 793.14 KiB | 7.55 MiB/s, done.
Resolving deltas: 100% (75/75), done.
Submodule path 'annotations': checked out 'a8a920a3293c4db270

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [10]:
# Dependencies: our repo + EgoVLP extras (einops, decord, timm, transformers, pytorchvideo)
!pip install -r /content/code/requirements.txt
!pip install -q einops timm transformers decord pytorchvideo ffmpeg
!pip install -q torcheval tqdm natsort opencv-python pillow scikit-learn

Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu118
  Preparing metadata (setup.py) ... done


In [4]:
# EgoVLP's FrozenInTime constructor loads a local ViT-B/16 checkpoint from
# /content/egovlp/pretrained/jx_vit_base_p16_224-80ecf9dd.pth. Download it.
!mkdir -p /content/egovlp/pretrained
!wget -q -nc -O /content/egovlp/pretrained/jx_vit_base_p16_224-80ecf9dd.pth \
  https://github.com/rwightman/pytorch-image-models/releases/download/v0.1-vitjx/jx_vit_base_p16_224-80ecf9dd.pth
!ls -la /content/egovlp/pretrained/

total 338192
drwxr-xr-x  2 root root      4096 Apr 28 16:05 .
drwxr-xr-x 13 root root      4096 Apr 28 16:05 ..
-rw-r--r--  1 root root 346292833 Dec  7  2021 jx_vit_base_p16_224-80ecf9dd.pth


In [12]:
# User-configurable paths on Drive
import os

DRIVE_ROOT = '/content/drive/MyDrive/AML_Project'
EGOVLP_CKPT = f'{DRIVE_ROOT}/models/egovlp.pth'
VIDEOS_DIR  = f'{DRIVE_ROOT}/CaptainCook4D/captain_cook_4d_gopro_resized_extracted'
FEATURES_DIR = f'{DRIVE_ROOT}/CaptainCook4D/features_egovlp_num_frames_16'
CKPT_DIR    = f'{DRIVE_ROOT}/models'

for p in [FEATURES_DIR, CKPT_DIR]:
    os.makedirs(p, exist_ok=True)

print('EGOVLP_CKPT exists:', os.path.exists(EGOVLP_CKPT))
print('VIDEOS_DIR  exists:', os.path.exists(VIDEOS_DIR))

# Export as env vars for bash/! cells
os.environ['EGOVLP_CKPT'] = EGOVLP_CKPT
os.environ['VIDEOS_DIR'] = VIDEOS_DIR
os.environ['FEATURES_DIR'] = FEATURES_DIR
os.environ['CKPT_DIR'] = CKPT_DIR

EGOVLP_CKPT exists: True
VIDEOS_DIR  exists: True


In [11]:
# Step 2: Extract EgoVLP features for every recording.
# Adapted from feature_extractors/frame_features/extract_features.py.
# Output: <FEATURES_DIR>/<recording_id>_360p.mp4_1s_1s.npz (shape [N, 256]).
%cd /content/code/core/features_extraction
!python segment_feature_extractor.py \
    --egovlp_repo /content/egovlp \
    --egovlp_ckpt "$EGOVLP_CKPT" \
    --videos_dir "$VIDEOS_DIR" \
    --output_dir "$FEATURES_DIR" \
    --backbone egovlp \
    --use_decord \
    --batch_size 16

/content/code/core/features_extraction
/usr/local/lib/python3.12/dist-packages/torchvision/transforms/_functional_video.py:6: UserWarning: The 'torchvision.transforms._functional_video' module is deprecated since 0.12 and will be removed in the future. Please use the 'torchvision.transforms.functional' module instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/transforms/_transforms_video.py:22: UserWarning: The 'torchvision.transforms._transforms_video' module is deprecated since 0.12 and will be removed in the future. Please use the 'torchvision.transforms' module instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
config.json: 100% 483/483 [00:00<00:00, 1.58MB/s]
model.safetensors: 100% 268M/268M [00:02<00:00, 11